# AI Design Benchmark — Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanJueLogic/MeiGen-DesignAgentBench/blob/main/examples/quickstart.ipynb)

This notebook reproduces the Round 1 results from the paper in ~2 minutes.

**What this notebook does:**
1. Clones the repository and loads the raw vote data
2. Runs the 4-layer aggregation to compute win rates per scene
3. Runs Bootstrap CI (95%) for overall win rates
4. Displays a summary chart

> No external dependencies beyond Python 3.9+ standard library.

In [ ]:
# Step 1: Clone the repo (skip if already cloned)
import os
if not os.path.exists('MeiGen-DesignAgentBench'):
    !git clone https://github.com/SanJueLogic/MeiGen-DesignAgentBench.git
os.chdir('MeiGen-DesignAgentBench')
print('Working directory:', os.getcwd())

In [ ]:
# Step 2: Load and inspect raw votes
import sys
sys.path.insert(0, 'tools')
from scoring import load_votes, aggregate

votes = load_votes('results/round-1/raw-votes.csv')
print(f'Total vote records: {len(votes)}')
print(f'Sample row: {votes[0]}')

In [ ]:
# Step 3: Run 4-layer aggregation
PRODUCT_MAP = {'1': 'Lovart', '2': 'Roboneo', '3': 'Jimeng (即梦)'}
results = aggregate(votes, PRODUCT_MAP)

# Print scene-level results
print(f'{'Scene':<30} {'Product':<16} {'Win Rate':>8} {'Tasks Won':>10}')
print('-' * 70)
for r in results:
    if r['scene'] != '[Total]':
        first_marker = ' ← 1st' if r['first_place'] == 'Yes' else ''
        print(f"{r['scene']:<30} {r['product']:<16} {r['win_rate']:>8} {r['tasks_won']:>8}/{r['total_tasks']}{first_marker}")

In [ ]:
# Step 4: Bootstrap 95% CI for overall win rates
from confidence import bootstrap_overall_ci

ci = bootstrap_overall_ci(votes, PRODUCT_MAP, n_iter=10_000, seed=42)
print('\n=== Overall Win Rate with 95% Bootstrap CI ===')
print(f"{'Product':<16} {'Win Rate':>9} {'95% CI'}")
print('-' * 45)
for prod, vals in sorted(ci.items(), key=lambda x: -x[1]['mean']):
    print(f"{prod:<16} {vals['mean']:>8.1%}  ({vals['lo']:.1%} – {vals['hi']:.1%})")

In [ ]:
# Step 5: Plot scene win rates (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    scenes = sorted(set(r['scene'] for r in results if r['scene'] != '[Total]'))
    products = list(PRODUCT_MAP.values())
    colors = {'Lovart': '#4C8EF5', 'Roboneo': '#F5A623', 'Jimeng (即梦)': '#7ED321'}

    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(scenes))
    width = 0.25

    scene_data = {s: {} for s in scenes}
    for r in results:
        if r['scene'] in scene_data:
            scene_data[r['scene']][r['product']] = float(r['win_rate'].strip('%')) / 100

    for i, prod in enumerate(products):
        rates = [scene_data[s].get(prod, 0) for s in scenes]
        bars = ax.bar([xi + i * width for xi in x], rates, width, label=prod,
                      color=colors.get(prod, '#999999'), alpha=0.85)

    ax.set_xlabel('Scene')
    ax.set_ylabel('Win Rate')
    ax.set_title('Round 1 Scene Win Rates — AI Design Benchmark')
    ax.set_xticks([xi + width for xi in x])
    ax.set_xticklabels([s.replace(' ', '\n') for s in scenes], fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.axhline(1/3, color='gray', linestyle='--', linewidth=0.8, label='Random baseline (33%)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('examples/round1_scene_winrates.png', dpi=150)
    plt.show()
    print('Chart saved to examples/round1_scene_winrates.png')
except ImportError:
    print('matplotlib not installed — skipping chart. Install with: pip install matplotlib')

## Next steps

- **Add a new product**: see [`docs/how-to-add-a-product.md`](../docs/how-to-add-a-product.md)
- **Run your own evaluation round**: see [`docs/how-to-run-a-round.md`](../docs/how-to-run-a-round.md)
- **Contribute new tasks**: see [`docs/how-to-contribute.md`](../docs/how-to-contribute.md)
- **Cite this work**: see [`CITATION.bib`](../CITATION.bib)